# Tokenization Math — Interactive Theory

Comprehensive implementation of every mathematical concept behind LLM tokenization.

**Contents:**
1. BPE from scratch (character-level and word-level)
2. Compression ratio analysis & diminishing returns
3. Unigram Language Model — full EM training with forward-backward
4. Token pruning via marginal loss (ΔL)
5. WordPiece — PMI-based merge criterion
6. Algorithm comparison: BPE vs Unigram vs WordPiece
7. Context window efficiency calculator
8. Vocabulary size & model parameter tradeoffs
9. Tokenization and arithmetic (digit splitting)
10. Fertility analysis
11. Special tokens
12. Information theory: Shannon entropy, Zipf's law, Rényi entropy
13. Compression vs Shannon bound
14. Viterbi DP segmentation (with full trace)
15. Number of possible segmentations (Fibonacci)
16. Real tokenizer comparison (tiktoken)

In [ ]:
import numpy as np
np.random.seed(42)
from collections import Counter, defaultdict
import re
import math

---
## 1. Byte Pair Encoding (BPE) — Complete Implementation

BPE is a greedy compression algorithm from information theory,
adapted for NLP by Sennrich et al. (2016). At each step:

$$(a^*, b^*) = \arg\max_{(a,b) \in V \times V} \sum_{i=1}^{|C|-1} \mathbb{1}[c_i = a \land c_{i+1} = b]$$

Each merge reduces corpus length by the pair's frequency:
$$|C_{\text{after}}| = |C_{\text{before}}| - f$$

In [ ]:
class BPE:
    """
    Full BPE implementation supporting both character-level and word-level
    pre-tokenization (as used by GPT-2/GPT-4).
    
    Character-level: treats entire text as one sequence of characters.
    Word-level: splits by whitespace/punctuation first, then applies BPE
                within each word. This prevents cross-word merges.
    """
    
    def __init__(self, word_level=False):
        self.merges = []          # ordered list of (pair, new_token)
        self.vocab = set()        # current vocabulary
        self.word_level = word_level
        self.word_freqs = None    # word → frequency (for word-level)
    
    # --- Core operations ---
    
    def _get_pairs(self, token_seq):
        """Count frequency of adjacent symbol pairs in a token sequence."""
        pairs = Counter()
        for i in range(len(token_seq) - 1):
            pairs[(token_seq[i], token_seq[i+1])] += 1
        return pairs
    
    def _get_pairs_wordlevel(self, word_splits):
        """Count pairs across all words, weighted by word frequency."""
        pairs = Counter()
        for word_tokens, freq in word_splits.items():
            for i in range(len(word_tokens) - 1):
                pairs[(word_tokens[i], word_tokens[i+1])] += freq
        return pairs
    
    def _merge_pair(self, token_seq, pair, new_token):
        """Replace all occurrences of (a, b) with new_token in a sequence."""
        result = []
        i = 0
        while i < len(token_seq):
            if i < len(token_seq) - 1 and (token_seq[i], token_seq[i+1]) == pair:
                result.append(new_token)
                i += 2
            else:
                result.append(token_seq[i])
                i += 1
        return result
    
    def _merge_pair_tuple(self, word_tuple, pair, new_token):
        """Merge pair in a tuple (hashable for dict keys)."""
        result = list(self._merge_pair(list(word_tuple), pair, new_token))
        return tuple(result)
    
    # --- Training ---
    
    def _pre_tokenize(self, text):
        """Split text into words (GPT-2 style: keep spaces attached to following word)."""
        # Simple regex: split on whitespace, keeping the space as prefix
        words = re.findall(r'\S+|\s+', text)
        word_freqs = Counter(words)
        # Convert each word to tuple of characters
        word_splits = {}
        for word, freq in word_freqs.items():
            word_splits[tuple(word)] = freq
        return word_splits
    
    def train(self, text, num_merges, verbose=True):
        """Train BPE tokenizer."""
        if self.word_level:
            return self._train_word_level(text, num_merges, verbose)
        else:
            return self._train_char_level(text, num_merges, verbose)
    
    def _train_char_level(self, text, num_merges, verbose):
        """Character-level BPE training."""
        tokens = list(text)
        self.vocab = set(tokens)
        
        if verbose:
            print(f'Initial: {len(tokens)} tokens, vocab={len(self.vocab)}')
        
        history = [{'step': 0, 'n_tokens': len(tokens), 'vocab_size': len(self.vocab),
                    'rho': len(text)/len(tokens)}]
        
        for step in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]
            new_token = best_pair[0] + best_pair[1]
            
            tokens = self._merge_pair(tokens, best_pair, new_token)
            self.vocab.add(new_token)
            self.merges.append((best_pair, new_token))
            
            rho = len(text) / len(tokens)
            history.append({'step': step+1, 'n_tokens': len(tokens),
                           'vocab_size': len(self.vocab), 'rho': rho})
            
            if verbose:
                print(f'Step {step+1:>3}: ("{best_pair[0]}","{best_pair[1]}") → "{new_token}"  '
                      f'freq={best_count}  tokens={len(tokens)}  ρ={rho:.2f}')
        
        return tokens, history
    
    def _train_word_level(self, text, num_merges, verbose):
        """Word-level BPE: pre-tokenize, then merge within words only."""
        word_splits = self._pre_tokenize(text)
        self.word_freqs = {w: f for w, f in Counter(re.findall(r'\S+|\s+', text)).items()}
        
        # Initialize vocab from all characters
        self.vocab = set()
        for word_tuple in word_splits:
            self.vocab.update(word_tuple)
        
        total_tokens = sum(len(w) * f for w, f in word_splits.items())
        if verbose:
            print(f'Initial: {total_tokens} tokens, {len(word_splits)} unique words, vocab={len(self.vocab)}')
        
        history = [{'step': 0, 'n_tokens': total_tokens, 'vocab_size': len(self.vocab),
                    'rho': len(text)/total_tokens}]
        
        for step in range(num_merges):
            pairs = self._get_pairs_wordlevel(word_splits)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]
            new_token = best_pair[0] + best_pair[1]
            
            # Apply merge to all words
            new_word_splits = {}
            for word_tuple, freq in word_splits.items():
                merged = self._merge_pair_tuple(word_tuple, best_pair, new_token)
                new_word_splits[merged] = freq
            word_splits = new_word_splits
            
            self.vocab.add(new_token)
            self.merges.append((best_pair, new_token))
            
            total_tokens = sum(len(w) * f for w, f in word_splits.items())
            rho = len(text) / total_tokens
            history.append({'step': step+1, 'n_tokens': total_tokens,
                           'vocab_size': len(self.vocab), 'rho': rho})
            
            if verbose:
                print(f'Step {step+1:>3}: ("{best_pair[0]}","{best_pair[1]}") → "{new_token}"  '
                      f'freq={best_count}  tokens={total_tokens}  ρ={rho:.2f}')
        
        self._word_splits = word_splits
        return word_splits, history
    
    def encode(self, text):
        """Encode text using learned merge rules."""
        if self.word_level:
            words = re.findall(r'\S+|\s+', text)
            all_tokens = []
            for word in words:
                tokens = list(word)
                for pair, new_token in self.merges:
                    tokens = self._merge_pair(tokens, pair, new_token)
                all_tokens.extend(tokens)
            return all_tokens
        else:
            tokens = list(text)
            for pair, new_token in self.merges:
                tokens = self._merge_pair(tokens, pair, new_token)
            return tokens

In [ ]:
# --- BPE Training Demo ---

corpus = "the cat sat on the mat the cat ate the rat the bat sat on the cat"

print('=== Character-Level BPE ===')
bpe_char = BPE(word_level=False)
tokens_char, history_char = bpe_char.train(corpus, num_merges=12)

print(f'\nFinal tokenization ({len(tokens_char)} tokens):')
print(tokens_char)

In [ ]:
# --- BPE Encoding and Lossless Reconstruction ---

test_texts = [
    "the cat sat",
    "the bat ate",
    "rat on mat",
    "unknown text here",  # out-of-training text
]

print(f'{"Text":<25} {"Tokens":>8} {"ρ":>6}  Tokenization')
print('-' * 80)
for text in test_texts:
    encoded = bpe_char.encode(text)
    reconstructed = ''.join(encoded)
    rho = len(text) / len(encoded)
    lossless = reconstructed == text
    print(f'{text:<25} {len(encoded):>8} {rho:>6.2f}  {encoded}')
    assert lossless, f'RECONSTRUCTION FAILED: "{reconstructed}" != "{text}"'

print(f'\n✓ All reconstructions are lossless (concat(tokens) == original text)')

In [ ]:
# --- Word-Level BPE (realistic, prevents cross-word merges) ---

print('=== Word-Level BPE ===')
bpe_word = BPE(word_level=True)
word_splits, history_word = bpe_word.train(corpus, num_merges=12)

print(f'\n--- Vocabulary after training ---')
for tok in sorted(bpe_word.vocab, key=len, reverse=True)[:15]:
    print(f'  "{tok}" (len={len(tok)})')

print(f'\n--- Encode with word-level BPE ---')
for text in ["the cat sat", "the bat ate", "unknown text"]:
    enc = bpe_word.encode(text)
    print(f'  "{text}" → {enc} ({len(enc)} tokens)')

print(f'\n→ Word-level BPE never merges across word boundaries')
print(f'→ "the cat" will never produce a merge spanning "e" and " " and "c"')
print(f'→ This is how GPT-2/GPT-4 actually work (with regex pre-tokenization)')

---
## 2. Compression Ratio Analysis

$$\rho = \frac{\text{length in bytes (or chars)}}{\text{length in tokens}}$$

Each merge reduces token count by the pair's frequency f.
Diminishing returns: early merges capture high-frequency patterns,
later merges capture increasingly rare ones.

In [ ]:
# Compression ratio vs number of merges — diminishing returns

large_corpus = (
    "the quick brown fox jumps over the lazy dog " * 50 +
    "she sells sea shells by the sea shore " * 30 +
    "how much wood would a woodchuck chuck " * 20 +
    "machine learning is transforming artificial intelligence " * 15
)

merge_counts = [0, 5, 10, 20, 30, 50, 80, 120]
results = []

for n_merges in merge_counts:
    bpe_test = BPE(word_level=False)
    tokens, hist = bpe_test.train(large_corpus, n_merges, verbose=False)
    final = hist[-1]
    results.append(final)

print(f'{"Merges":>8} {"Vocab":>8} {"Tokens":>10} {"ρ":>8} {"Marginal Δρ":>12}')
print('-' * 52)
prev_rho = 1.0
for mc, r in zip(merge_counts, results):
    delta = r['rho'] - prev_rho
    print(f'{mc:>8} {r["vocab_size"]:>8} {r["n_tokens"]:>10,} {r["rho"]:>8.2f} {delta:>+12.2f}')
    prev_rho = r['rho']

print(f'\n→ First 10 merges: Δρ ≈ {results[2]["rho"]-results[0]["rho"]:.2f} (massive gain)')
print(f'→ Merges 80→120: Δρ ≈ {results[7]["rho"]-results[6]["rho"]:.2f} (diminishing returns)')
print(f'→ Real tokenizers use 32k-128k merges but most compression comes from first few thousand')

---
## 3. Unigram Language Model — Full EM Training

The Unigram model (Kudo 2018) takes the opposite approach to BPE:
start with a large vocabulary, use EM to learn token probabilities,
then prune low-impact tokens.

**Objective**: maximize marginal log-likelihood:
$$\mathcal{L}(V) = \sum_{s \in D} \log \sum_{S \in \mathcal{S}(s)} \prod_{t \in S} P(t)$$

**E-step**: expected counts via forward-backward:
$$\alpha[j] = \text{logaddexp}_{i: s[i:j] \in V}(\alpha[i] + \log P(s[i:j]))$$

**M-step**:
$$P(t) \leftarrow \frac{E[\text{count}(t)]}{\sum_{t'} E[\text{count}(t')]}$$

In [ ]:
class UnigramLM:
    """
    Unigram Language Model tokenizer with EM training,
    forward-backward algorithm, and token pruning.
    
    This is mathematically equivalent to SentencePiece's Unigram mode.
    """
    
    def __init__(self, max_token_length=8):
        self.max_token_length = max_token_length
        self.vocab = {}      # token → log P(token)
        self.history = []    # training metrics
    
    # --- Vocabulary Initialization ---
    
    def _initialize_vocab(self, text, initial_vocab_size=300):
        """Initialize vocab from most frequent character n-grams."""
        # Count all substrings up to max_token_length
        substring_counts = Counter()
        for i in range(len(text)):
            for l in range(1, min(self.max_token_length + 1, len(text) - i + 1)):
                substring_counts[text[i:i+l]] += 1
        
        # Keep all single characters + top substrings
        chars = set(text)
        top_subs = [s for s, _ in substring_counts.most_common(initial_vocab_size)]
        all_tokens = list(set(top_subs) | chars)
        
        # Initialize with frequency-weighted probabilities
        total = sum(substring_counts[t] for t in all_tokens)
        for t in all_tokens:
            self.vocab[t] = np.log(substring_counts[t] / total)
    
    # --- Forward-Backward Algorithm ---
    
    def _forward(self, text):
        """
        Forward algorithm: α[j] = log Σ_{segmentations of text[0:j]} P(segmentation).
        Computed via log-sum-exp over all valid tokens ending at position j.
        """
        n = len(text)
        alpha = np.full(n + 1, -np.inf)
        alpha[0] = 0.0
        
        for j in range(1, n + 1):
            for i in range(max(0, j - self.max_token_length), j):
                substr = text[i:j]
                if substr in self.vocab:
                    alpha[j] = np.logaddexp(alpha[j], alpha[i] + self.vocab[substr])
        return alpha
    
    def _backward(self, text):
        """
        Backward algorithm: β[i] = log Σ_{segmentations of text[i:n]} P(segmentation).
        """
        n = len(text)
        beta = np.full(n + 1, -np.inf)
        beta[n] = 0.0
        
        for i in range(n - 1, -1, -1):
            for j in range(i + 1, min(i + self.max_token_length + 1, n + 1)):
                substr = text[i:j]
                if substr in self.vocab:
                    beta[i] = np.logaddexp(beta[i], self.vocab[substr] + beta[j])
        return beta
    
    # --- E-Step ---
    
    def _e_step(self, text):
        """
        Compute expected token counts via forward-backward.
        
        For token s[i:j], its posterior probability of being used is:
            P(use s[i:j]) = exp(α[i] + log P(s[i:j]) + β[j] - α[n])
        
        where α[n] = log P(text) = normalization constant.
        """
        alpha = self._forward(text)
        beta = self._backward(text)
        n = len(text)
        log_Z = alpha[n]  # total log-likelihood
        
        expected_counts = defaultdict(float)
        for i in range(n):
            for j in range(i + 1, min(i + self.max_token_length + 1, n + 1)):
                substr = text[i:j]
                if substr in self.vocab:
                    # posterior = exp(α[i] + log P(token) + β[j] - log Z)
                    log_posterior = alpha[i] + self.vocab[substr] + beta[j] - log_Z
                    expected_counts[substr] += np.exp(log_posterior)
        
        return dict(expected_counts), log_Z
    
    # --- M-Step ---
    
    def _m_step(self, expected_counts):
        """Update token log-probabilities from expected counts."""
        total = sum(expected_counts.values())
        for token in self.vocab:
            count = expected_counts.get(token, 1e-10)
            self.vocab[token] = np.log(count / total)
    
    # --- Viterbi Decoding ---
    
    def _viterbi(self, text):
        """Find the single best segmentation (MAP estimate)."""
        n = len(text)
        INF = float('inf')
        dp = [INF] * (n + 1)
        dp[0] = 0.0
        back = [-1] * (n + 1)
        
        for j in range(1, n + 1):
            for i in range(max(0, j - self.max_token_length), j):
                substr = text[i:j]
                if substr in self.vocab:
                    cost = -self.vocab[substr]  # minimize negative log-prob
                    if dp[i] + cost < dp[j]:
                        dp[j] = dp[i] + cost
                        back[j] = i
        
        if dp[n] == INF:
            return list(text)
        
        tokens = []
        j = n
        while j > 0:
            i = back[j]
            tokens.append(text[i:j])
            j = i
        return list(reversed(tokens))
    
    # --- Training ---
    
    def train(self, text, num_iterations=20, verbose=True):
        """Train via EM: alternate E-step and M-step."""
        self._initialize_vocab(text)
        
        if verbose:
            print(f'Initial vocab size: {len(self.vocab)}')
        
        for iteration in range(num_iterations):
            # E-step: compute expected counts
            expected_counts, log_likelihood = self._e_step(text)
            
            # M-step: update probabilities
            self._m_step(expected_counts)
            
            # Stats
            tokens = self._viterbi(text)
            self.history.append({
                'iter': iteration + 1,
                'LL': log_likelihood,
                'n_tokens': len(tokens),
                'vocab_size': len(self.vocab)
            })
            
            if verbose:
                print(f'  Iter {iteration+1:>3}: LL={log_likelihood:>10.2f}  '
                      f'tokens={len(tokens):>5}  vocab={len(self.vocab)}')
        
        return tokens
    
    # --- Token Pruning ---
    
    def prune(self, text, target_size, verbose=True):
        """
        Prune vocabulary by removing tokens with smallest marginal loss ΔL.
        
        ΔL(t) = L(V \ {t}) - L(V)
        
        Remove tokens with smallest ΔL (least impact on likelihood).
        Never remove single characters (to maintain coverage).
        """
        chars = set(text)
        
        while len(self.vocab) > target_size:
            baseline_alpha = self._forward(text)
            baseline_ll = baseline_alpha[len(text)]
            
            # Compute ΔL for each removable token
            delta_l = {}
            removable = [t for t in self.vocab if not (len(t) == 1 and t in chars)]
            
            for token in removable:
                saved_prob = self.vocab.pop(token)
                new_alpha = self._forward(text)
                new_ll = new_alpha[len(text)]
                delta_l[token] = baseline_ll - new_ll  # how much LL drops (≥ 0)
                self.vocab[token] = saved_prob
            
            if not delta_l:
                break
            
            # Remove token with smallest ΔL
            worst = min(delta_l, key=delta_l.get)
            del self.vocab[worst]
            
            if verbose and len(self.vocab) % 20 == 0:
                tokens = self._viterbi(text)
                print(f'  Pruned to {len(self.vocab)} tokens, '
                      f'removed "{worst}" (ΔL={delta_l[worst]:.4f}), '
                      f'segmentation: {len(tokens)} tokens')
        
        # Re-run a few EM iterations after pruning
        for _ in range(5):
            ec, _ = self._e_step(text)
            self._m_step(ec)
        
        if verbose:
            tokens = self._viterbi(text)
            print(f'  Final: vocab={len(self.vocab)}, tokens={len(tokens)}')
    
    def encode(self, text):
        """Encode text using current vocabulary (Viterbi)."""
        return self._viterbi(text)

In [ ]:
# --- Train Unigram model ---

corpus_uni = "the cat sat on the mat the cat ate the rat the bat sat on the cat"

print('=== Unigram Language Model Training (Full EM) ===\n')
unigram = UnigramLM(max_token_length=6)
tokens_uni = unigram.train(corpus_uni, num_iterations=15)

print(f'\nFinal tokenization:')
print(tokens_uni)
print(f'Tokens: {len(tokens_uni)}, ρ = {len(corpus_uni)/len(tokens_uni):.2f}')

In [ ]:
# --- EM Convergence Analysis ---

print('EM convergence (log-likelihood should increase monotonically):')
print(f'{"Iter":>5} {"Log-Likelihood":>16} {"ΔLL":>10} {"Tokens":>8}')
print('-' * 45)

prev_ll = None
for h in unigram.history:
    delta = h['LL'] - prev_ll if prev_ll is not None else 0.0
    print(f'{h["iter"]:>5} {h["LL"]:>16.4f} {delta:>+10.4f} {h["n_tokens"]:>8}')
    prev_ll = h['LL']

print(f'\n→ LL is non-decreasing (EM guarantee)')
print(f'→ ΔLL approaches 0 → convergence')
print(f'→ Token count stabilizes as probabilities converge')

---
## 4. Token Pruning via Marginal Loss (ΔL)

After EM converges, prune the vocabulary:
$$\Delta\mathcal{L}(t) = \mathcal{L}(V \setminus \{t\}) - \mathcal{L}(V) \geq 0$$

Remove tokens with smallest ΔL (least impact on corpus likelihood).
Never remove single characters (coverage guarantee).

In [ ]:
# --- Token Pruning Demo ---

print(f'Vocab before pruning: {len(unigram.vocab)} tokens')
print(f'Top tokens by probability:')
sorted_vocab = sorted(unigram.vocab.items(), key=lambda x: x[1], reverse=True)
for tok, logp in sorted_vocab[:15]:
    print(f'  "{tok}"  log P = {logp:.3f}  P = {np.exp(logp):.4f}')

# Prune to smaller vocab
target = max(40, len(set(corpus_uni)))  # at least keep all chars
print(f'\nPruning from {len(unigram.vocab)} → {target} tokens...')
unigram.prune(corpus_uni, target_size=target, verbose=True)

print(f'\nFinal tokenization after pruning:')
tokens_pruned = unigram.encode(corpus_uni)
print(tokens_pruned)
print(f'Tokens: {len(tokens_pruned)}, ρ = {len(corpus_uni)/len(tokens_pruned):.2f}')

print(f'\n→ Pruning removes tokens that contribute least to compression')
print(f'→ Single chars are never removed (guarantees any text can be encoded)')
print(f'→ This is the key difference from BPE: top-down (prune) vs bottom-up (grow)')

---
## 5. WordPiece — PMI-Based Merge Criterion

WordPiece (BERT) merges pairs by **pointwise mutual information**:

$$\text{PMI}(a, b) = \log \frac{P(a,b)}{P(a) \cdot P(b)} = \log \frac{\text{count}(ab) \cdot N}{\text{count}(a) \cdot \text{count}(b)}$$

High PMI = pair co-occurs much more than chance predicts.
BPE merges FREQUENT pairs; WordPiece merges SURPRISING pairs.

In [ ]:
# --- BPE (frequency) vs WordPiece (PMI) merge ranking ---

corpus_wp = "abcabcabcabc defdef ghighi abcdefabcdef xyxy"
tokens_wp = list(corpus_wp)

# Count pairs and individual tokens
pair_counts = Counter()
tok_counts = Counter(tokens_wp)
N_total = len(tokens_wp)

for i in range(len(tokens_wp) - 1):
    pair_counts[(tokens_wp[i], tokens_wp[i+1])] += 1

# BPE ranking: by raw frequency
print('=== BPE Ranking (by frequency) ===')
print(f'{"Pair":>12} {"Count":>8}')
for pair, count in pair_counts.most_common(10):
    print(f'{str(pair):>12} {count:>8}')

# WordPiece ranking: by PMI
print(f'\n=== WordPiece Ranking (by PMI) ===')
pmi_scores = {}
for (a, b), count_ab in pair_counts.items():
    p_ab = count_ab / (N_total - 1)
    p_a = tok_counts[a] / N_total
    p_b = tok_counts[b] / N_total
    pmi = np.log2(p_ab / (p_a * p_b)) if p_a > 0 and p_b > 0 else 0
    pmi_scores[(a, b)] = pmi

print(f'{"Pair":>12} {"PMI":>8} {"Count":>8} {"P(a)":>8} {"P(b)":>8}')
for pair, pmi in sorted(pmi_scores.items(), key=lambda x: -x[1])[:10]:
    a, b = pair
    print(f'{str(pair):>12} {pmi:>8.2f} {pair_counts[pair]:>8} '
          f'{tok_counts[a]/N_total:>8.3f} {tok_counts[b]/N_total:>8.3f}')

print(f'\n→ BPE picks the globally most FREQUENT pair')
print(f'→ WordPiece picks pairs that are most SURPRISINGLY co-occurring')
print(f'→ Rare pairs that ALWAYS appear together get high PMI even with low count')
print(f'→ PMI = log(P(ab)/P(a)P(b)): dividing by P(a)P(b) normalizes for base frequency')

---
## 6. Algorithm Comparison: BPE vs Unigram vs WordPiece

All three algorithms on the **same corpus**, showing how they produce
different segmentations.

In [ ]:
# --- Side-by-side comparison ---

comparison_corpus = "the cat sat on the mat the cat ate the rat"
test_string = "the cat sat"

# 1. BPE
bpe_cmp = BPE(word_level=False)
bpe_cmp.train(comparison_corpus, num_merges=10, verbose=False)
bpe_result = bpe_cmp.encode(test_string)

# 2. Unigram
uni_cmp = UnigramLM(max_token_length=6)
uni_cmp.train(comparison_corpus, num_iterations=15, verbose=False)
uni_result = uni_cmp.encode(test_string)

# 3. WordPiece (simulate: BPE but merge by PMI instead of frequency)
class WordPieceBPE(BPE):
    """BPE variant that merges by PMI instead of frequency."""
    def _train_char_level(self, text, num_merges, verbose):
        tokens = list(text)
        self.vocab = set(tokens)
        history = [{'step': 0, 'n_tokens': len(tokens), 'vocab_size': len(self.vocab),
                    'rho': len(text)/len(tokens)}]
        
        for step in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            tok_counts = Counter(tokens)
            N = len(tokens)
            
            # Score by PMI instead of raw frequency
            pmi = {}
            for (a, b), c_ab in pairs.items():
                p_ab = c_ab / (N - 1)
                p_a = tok_counts[a] / N
                p_b = tok_counts[b] / N
                pmi[(a, b)] = np.log2(p_ab / (p_a * p_b)) if p_a > 0 and p_b > 0 else -np.inf
            
            best_pair = max(pmi, key=pmi.get)
            new_token = best_pair[0] + best_pair[1]
            tokens = self._merge_pair(tokens, best_pair, new_token)
            self.vocab.add(new_token)
            self.merges.append((best_pair, new_token))
            
            rho = len(text) / len(tokens)
            history.append({'step': step+1, 'n_tokens': len(tokens),
                           'vocab_size': len(self.vocab), 'rho': rho})
        return tokens, history

wp_cmp = WordPieceBPE(word_level=False)
wp_cmp.train(comparison_corpus, num_merges=10, verbose=False)
wp_result = wp_cmp.encode(test_string)

print(f'Corpus: "{comparison_corpus}"')
print(f'Test:   "{test_string}"\n')
print(f'{"Algorithm":<12} {"Tokens":>8} {"ρ":>6}  Segmentation')
print('-' * 65)
print(f'{"BPE":<12} {len(bpe_result):>8} {len(test_string)/len(bpe_result):>6.2f}  {bpe_result}')
print(f'{"Unigram":<12} {len(uni_result):>8} {len(test_string)/len(uni_result):>6.2f}  {uni_result}')
print(f'{"WordPiece":<12} {len(wp_result):>8} {len(test_string)/len(wp_result):>6.2f}  {wp_result}')

print(f'\n→ Different algorithms, different segmentations')
print(f'→ BPE: deterministic greedy merges by frequency')
print(f'→ Unigram: probabilistic optimal segmentation via Viterbi')
print(f'→ WordPiece: greedy merges by PMI (surprise, not frequency)')

---
## 7. Context Window Efficiency

The context window is measured in **tokens**, not characters.
Compression ratio ρ directly determines how much text fits.

In [ ]:
# --- Context Window Calculator ---

context_sizes = [2048, 4096, 8192, 32768, 131072]  # common LLM context windows
rho_values = [
    ('Poor (CJK, bad tokenizer)', 2.0),
    ('Average (multilingual)', 3.0),
    ('Good (English, BPE)', 3.7),
    ('Excellent (English, optimized)', 4.2),
]

print(f'{"":<30}', end='')
for ctx in context_sizes:
    if ctx >= 1000:
        label = f'{ctx//1024}K'
    else:
        label = str(ctx)
    print(f'{label:>10}', end='')
print()
print('-' * (30 + 10 * len(context_sizes)))

for name, rho in rho_values:
    print(f'{name:<30}', end='')
    for ctx in context_sizes:
        chars = int(ctx * rho)
        if chars >= 1_000_000:
            label = f'{chars/1_000_000:.1f}M'
        elif chars >= 1000:
            label = f'{chars/1000:.0f}K'
        else:
            label = str(chars)
        print(f'{label:>10}', end='')
    print()

print(f'\n→ Improving ρ from 2.0 to 4.0 DOUBLES the effective context length')
print(f'→ For 128K context: ρ=2.0 gives ~256K chars, ρ=4.0 gives ~512K chars')
print(f'→ CJK users effectively get HALF the context window of English users')

# Concrete example: a book
avg_book_chars = 500_000
print(f'\nExample: average novel ≈ {avg_book_chars:,} characters')
for name, rho in rho_values:
    tokens_needed = int(avg_book_chars / rho)
    fits_128k = '✓' if tokens_needed <= 131072 else '✗'
    print(f'  {name:<30}: {tokens_needed:>8,} tokens needed  [{fits_128k} fits in 128K]')

---
## 8. Vocabulary Size & Model Parameter Tradeoffs

Embedding: $E \in \mathbb{R}^{N \times d}$ → $N \times d$ params

LM Head: $W \in \mathbb{R}^{d \times N}$ → $d \times N$ params

Weight tying ($E = W^T$) halves the cost.

In [ ]:
# --- Vocabulary parameter cost ---

models = [
    ('GPT-2 Small',   768,   124_000_000, 50_257),
    ('GPT-2 Large',  1280,   774_000_000, 50_257),
    ('LLaMA-7B',     4096, 7_000_000_000, 32_000),
    ('LLaMA-13B',    5120, 13_000_000_000, 32_000),
    ('LLaMA-3 8B',   4096, 8_000_000_000, 128_000),
    ('GPT-4 (est.)', 8192, 200_000_000_000, 100_000),
]

print(f'{"Model":<16} {"d":>6} {"V":>8} {"Embed":>12} {"% (untied)":>12} {"% (tied)":>10}')
print('-' * 68)

for name, d, total, V in models:
    embed_params = V * d
    untied_pct = (2 * embed_params) / total * 100
    tied_pct = embed_params / total * 100
    print(f'{name:<16} {d:>6} {V:>8,} {embed_params:>12,} {untied_pct:>11.1f}% {tied_pct:>9.1f}%')

print(f'\n→ GPT-2 Small: embeddings are {2*50257*768/124_000_000:.0%} of all params (untied)!')
print(f'→ LLaMA-3 8B (V=128K): even with tying, embeddings are {128000*4096/8e9:.1%} of model')
print(f'→ Weight tying is standard: saves half the vocab parameters')

In [ ]:
# --- Memory footprint of embedding table ---

print('Embedding table memory (float16 = 2 bytes, float32 = 4 bytes):')
print(f'{"Vocab":>10} {"d_model":>10} {"FP16 (MB)":>12} {"FP32 (MB)":>12} {"FP16 (GB)":>12}')
print('-' * 60)

for V in [32_000, 50_257, 128_000, 256_000]:
    for d in [768, 4096, 8192]:
        fp16_bytes = V * d * 2
        fp32_bytes = V * d * 4
        fp16_mb = fp16_bytes / (1024**2)
        fp32_mb = fp32_bytes / (1024**2)
        fp16_gb = fp16_bytes / (1024**3)
        print(f'{V:>10,} {d:>10} {fp16_mb:>12.1f} {fp32_mb:>12.1f} {fp16_gb:>12.3f}')
    print()

print(f'→ V=256K, d=8192 in FP16: {256000*8192*2/1024**3:.1f} GB just for embeddings')
print(f'→ On edge devices (phones), this can exceed available memory')
print(f'→ Motivates: quantized embeddings, vocabulary pruning, smaller V')

---
## 9. Tokenization and Arithmetic (Digit Splitting)

Inconsistent number tokenization is a key reason LLMs fail at arithmetic.
The model must learn that `"4"` as a suffix has different positional
meaning than `"4"` standalone.

In [ ]:
# --- Digit splitting analysis ---

# Train BPE on a corpus that includes some numbers
arithmetic_corpus = (
    "the value is 100 and the count is 200 the total is 300 " * 20 +
    "12 plus 34 equals 46 and 56 plus 78 equals 134 " * 15 +
    "price is 1234 or maybe 5678 or even 91011 dollars " * 10
)

bpe_arith = BPE(word_level=False)
bpe_arith.train(arithmetic_corpus, num_merges=30, verbose=False)

# Test how different numbers get tokenized
test_numbers = ['1', '12', '123', '1234', '12345', '123456',
                '100', '200', '300', '999', '1000', '10000',
                '42', '420', '4200']

print('How numbers get tokenized (inconsistently!):')
print(f'{"Number":<10} {"Tokens":>8} {"Segmentation":<30} {"Consistent?"}')
print('-' * 65)

for num in test_numbers:
    tokens = bpe_arith.encode(num)
    # Check if each digit maps to its own token
    all_single = all(len(t) == 1 for t in tokens)
    one_token = len(tokens) == 1
    status = 'all-single' if all_single else ('merged' if one_token else 'MIXED')
    print(f'{num:<10} {len(tokens):>8}  {str(tokens):<30} {status}')

print(f'\n→ "100" and "200" might merge to single tokens (frequent in training)')
print(f'→ "1234" might split as ["12","34"] or ["1","234"] depending on corpus')
print(f'→ This inconsistency means the model has no reliable positional encoding for digits')
print(f'→ Addition requires knowing: "3" in "123" is hundreds place; hard when split varies')
print(f'\nFix in practice: some models force single-digit tokenization (each digit = 1 token)')

---
## 10. Tokenization Fertility

$$\text{fertility}(w) = |T(w)|$$

Average tokens per word. High fertility = more compute,
harder to learn representations, higher cost.

In [ ]:
# --- Fertility Analysis ---

# Train a tokenizer on English-heavy corpus
english_corpus = (
    "the quick brown fox jumps over the lazy dog " * 40 +
    "machine learning is transforming artificial intelligence " * 30 +
    "deep neural networks learn hierarchical representations from data " * 20 +
    "the cat sat on the mat and ate the rat and the bat " * 30 +
    "natural language processing enables computers to understand text " * 15
)

bpe_fert = BPE(word_level=False)
bpe_fert.train(english_corpus, num_merges=50, verbose=False)

def compute_fertility(text, tokenizer, split_by=' '):
    """Compute per-word fertility: tokens_per_word."""
    words = text.split(split_by)
    words = [w for w in words if w]  # remove empty
    fertilities = []
    for word in words:
        tokens = tokenizer.encode(word)
        fertilities.append(len(tokens))
    return words, fertilities

# Test on different text types
test_cases = {
    'English (in-domain)':    'the quick brown fox jumps over the lazy dog',
    'English (out-domain)':   'cryptocurrency regulations affect global markets significantly',
    'Technical':              'backpropagation gradient descent optimization stochastic convergence',
    'Mixed case':             'JavaScript TypeScript PostgreSQL Kubernetes',
    'German-ish':             'unbelievable kindergarten wanderlust zeitgeist',
}

print(f'{"Text Type":<25} {"Avg Fert":>10} {"Max":>6} {"Words":>7} {"Tokens":>8}')
print('-' * 60)

for name, text in test_cases.items():
    words, ferts = compute_fertility(text, bpe_fert)
    avg_f = np.mean(ferts)
    max_f = max(ferts)
    total_t = sum(ferts)
    print(f'{name:<25} {avg_f:>10.2f} {max_f:>6} {len(words):>7} {total_t:>8}')

# Per-word detail for one example
print(f'\nPer-word fertility (out-of-domain):')
words, ferts = compute_fertility(test_cases['English (out-domain)'], bpe_fert)
for word, f in zip(words, ferts):
    bar = '█' * f
    print(f'  {word:<20} → {f} token(s)  {bar}')

print(f'\n→ In-domain words: low fertility (tokenizer learned these patterns)')
print(f'→ Out-of-domain words: higher fertility (fragmented into small pieces)')
print(f'→ This reveals tokenizer\'s training data distribution')

---
## 11. Special Tokens

Special tokens carry structural information and are **manually added**
to the vocabulary (not learned by BPE/Unigram).

In [ ]:
# --- Special Tokens Demo ---

special_tokens = {
    '<|bos|>': {'id': 1,   'purpose': 'Beginning of sequence',
                'usage': 'Prepended to every input. Signals "start generating".'},
    '<|eos|>': {'id': 2,   'purpose': 'End of sequence',
                'usage': 'Model generates this to signal it is done. Also used as stop token.'},
    '<|pad|>': {'id': 0,   'purpose': 'Padding for batching',
                'usage': 'Fills shorter sequences in a batch to match longest. Masked in attention.'},
    '<|unk|>': {'id': 3,   'purpose': 'Unknown token',
                'usage': 'Fallback for byte-level failures. Rare in BPE (which covers all bytes).'},
    '<|sep|>': {'id': 102, 'purpose': 'Segment separator (BERT)',
                'usage': 'Separates sentence A from sentence B in BERT-style models.'},
}

print(f'{"Token":<12} {"ID":>5} {"Purpose":<25} Usage')
print('-' * 85)
for token, info in special_tokens.items():
    print(f'{token:<12} {info["id"]:>5} {info["purpose"]:<25} {info["usage"]}')

# Demonstrate how special tokens affect sequence construction
print(f'\n--- Example: Constructing a model input ---')
text = "Hello, how are you?"
tokens = bpe_char.encode(text)
token_ids = list(range(1, len(tokens) + 1))  # dummy IDs

full_sequence = ['<|bos|>'] + tokens + ['<|eos|>']
full_ids = [1] + token_ids + [2]

print(f'Text:     "{text}"')
print(f'Tokens:   {tokens}')
print(f'With special: {full_sequence}')
print(f'IDs:      {full_ids}')

# Batching with padding
print(f'\n--- Batching (sequences of different lengths) ---')
seq1 = ['<|bos|>', 'hello', '<|eos|>']
seq2 = ['<|bos|>', 'how', 'are', 'you', '<|eos|>']
max_len = max(len(seq1), len(seq2))
padded1 = seq1 + ['<|pad|>'] * (max_len - len(seq1))
padded2 = seq2
print(f'Seq 1: {padded1}')
print(f'Seq 2: {padded2}')
print(f'Attention mask 1: {[1]*len(seq1) + [0]*(max_len-len(seq1))}')
print(f'Attention mask 2: {[1]*len(seq2)}')
print(f'\n→ <|pad|> tokens are MASKED in attention (the model ignores them)')
print(f'→ Special tokens are added to V manually, not learned by BPE')

---
## 12. Information Theory: Shannon Entropy & Zipf's Law

### Shannon Entropy
$$H_{\text{token}} = -\sum_{t \in V} P(t) \log_2 P(t)$$

Maximum: $H_{\max} = \log_2 |V|$ (uniform distribution).

Efficiency: $\eta = H / H_{\max}$

### Zipf's Law
Token frequency follows a power law: $f(r) \propto r^{-\alpha}$
where $r$ is rank and $\alpha \approx 1$.

In [ ]:
# --- Shannon Entropy Analysis ---

def full_entropy_analysis(tokens, label=''):
    """
    Complete information-theoretic analysis of a token distribution.
    Returns dict of metrics.
    """
    counts = Counter(tokens)
    total = sum(counts.values())
    V = len(counts)
    probs = np.array(sorted([c/total for c in counts.values()], reverse=True))
    
    # Shannon entropy
    H = -np.sum(probs * np.log2(probs))
    H_max = np.log2(V) if V > 1 else 0
    efficiency = H / H_max if H_max > 0 else 0
    
    # Cross-entropy (bits per token in optimal code)
    # If we used Huffman coding, each token needs ~H bits
    
    # Perplexity (effective vocabulary size)
    perplexity = 2 ** H
    
    result = {
        'V': V, 'N': total, 'H': H, 'H_max': H_max,
        'efficiency': efficiency, 'perplexity': perplexity,
        'probs': probs
    }
    
    if label:
        print(f'=== {label} ===')
        print(f'  Unique tokens (V):     {V}')
        print(f'  Total tokens (N):      {total:,}')
        print(f'  Shannon entropy H:     {H:.3f} bits/token')
        print(f'  Max entropy H_max:     {H_max:.3f} bits (= log₂({V}))')
        print(f'  Efficiency η = H/H_max: {efficiency:.2%}')
        print(f'  Perplexity 2^H:        {perplexity:.1f} (effective V)')
    
    return result


# Compare character-level vs BPE
text_analysis = (
    "the quick brown fox jumps over the lazy dog " * 30 +
    "she sells sea shells by the sea shore " * 20
)

# Character-level
char_tokens = list(text_analysis)
r1 = full_entropy_analysis(char_tokens, 'Character-level')
print()

# BPE with 30 merges
bpe_entropy = BPE(word_level=False)
bpe_tokens, _ = bpe_entropy.train(text_analysis, 30, verbose=False)
r2 = full_entropy_analysis(bpe_tokens, f'BPE (30 merges)')
print()

# BPE with 60 merges
bpe_entropy2 = BPE(word_level=False)
bpe_tokens2, _ = bpe_entropy2.train(text_analysis, 60, verbose=False)
r3 = full_entropy_analysis(bpe_tokens2, f'BPE (60 merges)')

print(f'\n→ BPE increases entropy per token (each token carries more information)')
print(f'→ But efficiency drops: many BPE tokens are rare (low utilization)')
print(f'→ Perplexity = effective vocabulary size: char ≈ {r1["perplexity"]:.0f}, '
      f'BPE-30 ≈ {r2["perplexity"]:.0f}, BPE-60 ≈ {r3["perplexity"]:.0f}')

In [ ]:
# --- Zipf's Law in Token Distributions ---

counts = Counter(bpe_tokens)
sorted_items = counts.most_common()
total = sum(c for _, c in sorted_items)

print('Token frequency distribution (Zipf\'s law):')
print(f'{"Rank":>6} {"Token":>15} {"Count":>8} {"Freq":>8} {"Cumul":>8}')
print('-' * 52)

cumulative = 0
for rank, (token, count) in enumerate(sorted_items[:20], 1):
    freq = count / total
    cumulative += freq
    bar = '█' * int(freq * 60)
    print(f'{rank:>6} {repr(token):>15} {count:>8} {freq:>7.3f} {cumulative:>7.1%} {bar}')

# Verify power law: log(freq) vs log(rank) should be roughly linear
ranks = np.arange(1, len(sorted_items) + 1)
freqs = np.array([c/total for _, c in sorted_items])

# Fit power law: log f = -α log r + c
log_r = np.log(ranks)
log_f = np.log(freqs)
alpha, c = np.polyfit(log_r, log_f, 1)

print(f'\nZipf\'s α = {-alpha:.2f} (expected ≈ 1.0 for natural language)')
print(f'Top 5 tokens: {cumulative:.0%} of all tokens' if rank >= 5 else '')
print(f'Top 50% of tokens appear only {sum(1 for c in freqs if c < np.median(freqs))} times')
print(f'\n→ Token distributions are highly skewed (Zipfian)')
print(f'→ A few tokens dominate; most tokens are rare')
print(f'→ This is why entropy < H_max: the distribution is far from uniform')

---
## 13. Rényi Entropy Suite

The Rényi entropy of order α generalizes Shannon entropy:

$$H_\alpha = \frac{1}{1-\alpha} \log_2 \left(\sum_{t \in V} P(t)^\alpha\right)$$

| α | Name | What it measures |
|---|------|------------------|
| α → 1 | Shannon entropy | Average surprise |
| α = 2 | Collision entropy | Probability two random tokens match |
| α → ∞ | Min-entropy | Dominated by most frequent token |

Large $H_1 - H_2$ gap → heavy tail of rarely-used tokens.

In [ ]:
# --- Rényi Entropy Suite ---

def renyi_entropy(probs, alpha):
    """
    Compute Rényi entropy of order α.
    
    Special cases:
      α → 1: Shannon entropy (computed via limit)
      α = 2: collision entropy = -log₂(Σ pᵢ²)
      α → ∞: min-entropy = -log₂(max pᵢ)
    """
    probs = np.array(probs)
    probs = probs[probs > 0]  # remove zeros
    
    if np.isclose(alpha, 1.0):
        # Shannon entropy (limit as α → 1)
        return -np.sum(probs * np.log2(probs))
    elif np.isinf(alpha):
        # Min-entropy
        return -np.log2(np.max(probs))
    else:
        return (1 / (1 - alpha)) * np.log2(np.sum(probs ** alpha))


# Analyze BPE token distribution
counts = Counter(bpe_tokens)
total = sum(counts.values())
probs = np.array([c/total for c in counts.values()])

alphas = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, np.inf]
alpha_labels = ['0.5', '1.0 (Shannon)', '1.5', '2.0 (Collision)',
                '3.0', '5.0', '10.0', '∞ (Min-entropy)']

H_max = np.log2(len(counts))

print(f'Rényi entropy spectrum for BPE token distribution (V={len(counts)}):')
print(f'H_max = log₂({len(counts)}) = {H_max:.3f} bits\n')

print(f'{"α":>20} {"H_α (bits)":>12} {"H_α / H_max":>12} {"2^H_α":>12}')
print('-' * 60)

H_values = []
for alpha, label in zip(alphas, alpha_labels):
    H = renyi_entropy(probs, alpha)
    H_values.append(H)
    ratio = H / H_max if H_max > 0 else 0
    print(f'{label:>20} {H:>12.3f} {ratio:>12.2%} {2**H:>12.1f}')

H1 = renyi_entropy(probs, 1.0)
H2 = renyi_entropy(probs, 2.0)
H_inf = renyi_entropy(probs, np.inf)

print(f'\nKey relationships:')
print(f'  H_∞ ≤ H_2 ≤ H_1 ≤ H_0.5 ≤ H_max  (always, for any distribution)')
print(f'  H_1 - H_2 gap: {H1 - H2:.3f} bits')
print(f'  H_1 - H_∞ gap: {H1 - H_inf:.3f} bits')
print(f'\n→ Large H₁-H₂ gap ({H1-H2:.2f}) indicates heavy tail of rare tokens')
print(f'→ The vocabulary has many tokens that are almost never used')
print(f'→ Uniform distribution would give H₁ = H₂ = H_∞ = {H_max:.2f} (no gap)')

In [ ]:
# --- Compare: uniform vs skewed vs BPE token distributions ---

distributions = {
    'Uniform (8 tokens)': np.ones(8) / 8,
    'Moderate skew':      np.array([0.3, 0.2, 0.15, 0.1, 0.1, 0.08, 0.05, 0.02]),
    'Heavy skew':         np.array([0.6, 0.15, 0.1, 0.05, 0.04, 0.03, 0.02, 0.01]),
    'BPE (actual)':       probs,
}

print(f'{"Distribution":<22} {"H₁ (Shannon)":>14} {"H₂ (Collision)":>16} {"H_∞ (Min)":>12} {"H₁-H₂ gap":>12}')
print('-' * 80)

for name, p in distributions.items():
    h1 = renyi_entropy(p, 1.0)
    h2 = renyi_entropy(p, 2.0)
    h_inf = renyi_entropy(p, np.inf)
    gap = h1 - h2
    print(f'{name:<22} {h1:>14.3f} {h2:>16.3f} {h_inf:>12.3f} {gap:>12.3f}')

print(f'\n→ Uniform: H₁ = H₂ = H_∞ (zero gap, perfect balance)')
print(f'→ As skew increases, gap widens (more wasted vocab entries)')
print(f'→ BPE token distributions are typically heavily skewed (Zipfian)')

---
## 14. Compression vs Shannon Bound

The entropy rate of English is approximately 1.0–1.5 bits/character (Shannon's experiments).

A tokenizer achieves:
$$\text{bits/char} = \frac{H_{\text{token}}}{\rho}$$

where H is the actual token entropy and ρ is compression ratio.
If using uniform encoding: bits/char = log₂(V) / ρ.

In [ ]:
# --- Compression efficiency vs Shannon bound ---

print('Tokenizer compression vs Shannon entropy bound for English:\n')
shannon_bound = 1.3  # bits/char (Shannon's estimate for English)

scenarios = [
    ('Character-level (ASCII)',    1.0,  np.log2(256)),
    ('Character-level (letters)',  1.0,  np.log2(27)),  # 26 letters + space
    ('Word-level (10K words)',     5.0,  np.log2(10000)),
    ('BPE 32K (uniform code)',     3.7,  np.log2(32000)),
    ('BPE 32K (Huffman-like)',     3.7,  10.0),   # actual token entropy ≈ 10 bits
    ('BPE 50K (GPT-2 style)',      3.9,  np.log2(50257)),
    ('BPE 100K (GPT-4 style)',     4.0,  np.log2(100000)),
    ('BPE 100K (Huffman-like)',    4.0,  11.0),
]

print(f'{"Tokenizer":<30} {"ρ":>6} {"bits/token":>12} {"bits/char":>12} {"vs Shannon":>12}')
print('-' * 76)

for name, rho, bits_tok in scenarios:
    bits_char = bits_tok / rho
    overhead = bits_char / shannon_bound
    print(f'{name:<30} {rho:>6.1f} {bits_tok:>12.1f} {bits_char:>12.2f} {overhead:>11.1f}x')

print(f'\nShannon bound: {shannon_bound} bits/char')
print(f'\n→ Uniform encoding (log₂V) wastes bits because token distribution is NOT uniform')
print(f'→ Actual token entropy (Huffman-like) is much lower than log₂V')
print(f'→ BPE with Huffman-like coding gets within ~2x of Shannon bound')
print(f'→ The language model itself implicitly learns to approach Shannon\'s bound')
print(f'→ GPT-4\'s cross-entropy ≈ 1.5-2.0 bits/char on English text (near-optimal!)')

---
## 15. Viterbi DP Segmentation (with Full Trace)

$$\text{dp}[j] = \min_{i: s[i:j] \in V} \left(\text{dp}[i] + w(s[i:j])\right)$$

Complexity: $O(n \cdot L)$ where $L$ = max token length.

In [ ]:
# --- Viterbi DP with full step-by-step trace ---

def viterbi_tokenize(text, vocab_costs, verbose=False):
    """
    Find optimal segmentation via dynamic programming.
    
    Args:
        text: string to segment
        vocab_costs: dict mapping token → cost (= -log P)
        verbose: print step-by-step trace
    
    Returns:
        (total_cost, token_list)
    """
    n = len(text)
    INF = float('inf')
    dp = [INF] * (n + 1)
    dp[0] = 0.0
    backtrack = [-1] * (n + 1)
    max_len = max(len(t) for t in vocab_costs)
    
    if verbose:
        print(f'Text: "{text}" (n={n})')
        print(f'Vocab: {vocab_costs}')
        print(f'Max token length: {max_len}')
        print(f'dp[0] = 0.0 (base case)\n')
    
    for j in range(1, n + 1):
        if verbose:
            print(f'--- j={j} (filling dp[{j}]) ---')
        
        for i in range(max(0, j - max_len), j):
            substr = text[i:j]
            if substr in vocab_costs:
                candidate = dp[i] + vocab_costs[substr]
                better = candidate < dp[j]
                if verbose:
                    marker = ' ← NEW BEST' if better else ''
                    print(f'  s[{i}:{j}]="{substr}" (cost {vocab_costs[substr]:.1f}): '
                          f'dp[{i}]={dp[i]:.1f} + {vocab_costs[substr]:.1f} = {candidate:.1f}{marker}')
                if better:
                    dp[j] = candidate
                    backtrack[j] = i
        
        if verbose:
            print(f'  → dp[{j}] = {dp[j]:.1f}\n')
    
    # Reconstruct
    if dp[n] == INF:
        return INF, None
    
    tokens = []
    j = n
    while j > 0:
        i = backtrack[j]
        tokens.append(text[i:j])
        j = i
    tokens.reverse()
    
    if verbose:
        print(f'=== Backtrack ===')
        print(f'dp table: {[f"{x:.1f}" if x < INF else "∞" for x in dp]}')
        print(f'back:     {backtrack}')
        print(f'Optimal:  {tokens} (cost={dp[n]:.1f})')
    
    return dp[n], tokens


# Worked example: full trace
vocab_costs = {'a': 1.0, 'b': 1.5, 'ab': 0.5, 'ba': 0.8}
cost, tokens = viterbi_tokenize('abab', vocab_costs, verbose=True)

In [ ]:
# --- Enumerate ALL segmentations and verify Viterbi finds optimal ---

def enumerate_segmentations(text, vocab):
    """Recursively enumerate all valid segmentations."""
    if not text:
        return [[]]
    results = []
    for end in range(1, len(text) + 1):
        prefix = text[:end]
        if prefix in vocab:
            for rest in enumerate_segmentations(text[end:], vocab):
                results.append([prefix] + rest)
    return results

text = 'abab'
all_segs = enumerate_segmentations(text, vocab_costs)

print(f'All {len(all_segs)} valid segmentations of "{text}":')
print(f'{"Segmentation":<25} {"Cost":<10} {"Optimal?"}')
print('-' * 45)

for seg in sorted(all_segs, key=lambda s: sum(vocab_costs[t] for t in s)):
    total_cost = sum(vocab_costs[t] for t in seg)
    optimal = '← optimal' if abs(total_cost - cost) < 1e-9 else ''
    cost_str = ' + '.join(f'{vocab_costs[t]:.1f}' for t in seg)
    print(f'{str(seg):<25} {cost_str} = {total_cost:.1f}  {optimal}')

print(f'\n✓ Viterbi gives same answer as brute-force ({len(all_segs)} segmentations checked)')

In [ ]:
# --- Realistic example: segmenting an English word ---

english_vocab = {
    # Single characters (expensive fallback)
    'u': 4.0, 'n': 3.5, 'b': 4.0, 'e': 3.0, 'l': 3.5,
    'i': 3.0, 'v': 4.0, 'a': 3.0,
    # Common subwords
    'un': 2.0, 'be': 2.5, 'in': 2.0, 'le': 2.5,
    'able': 1.5, 'lieve': 2.0, 'believ': 1.8,
    # Full morphemes
    'believe': 1.2, 'unbeliev': 1.5,
    'unbelievable': 0.8,
}

word = 'unbelievable'
cost_full, tokens_full = viterbi_tokenize(word, english_vocab)
print(f'Full vocab:    {tokens_full} (cost={cost_full:.1f})')

# What if the full word isn't in vocab?
limited = {k: v for k, v in english_vocab.items() if len(k) < 10}
cost_lim, tokens_lim = viterbi_tokenize(word, limited)
print(f'Without full:  {tokens_lim} (cost={cost_lim:.1f})')

# What about character-only?
chars_only = {k: v for k, v in english_vocab.items() if len(k) == 1}
cost_char, tokens_char = viterbi_tokenize(word, chars_only)
print(f'Chars only:    {tokens_char} (cost={cost_char:.1f})')

print(f'\n→ More subwords in vocab = fewer tokens = lower cost')
print(f'→ But each extra vocab entry costs {4096} embedding parameters (d=4096)')
print(f'→ Cost of "unbelievable" as 1 token: {0.8:.1f}.  As 12 chars: {cost_char:.1f}')
print(f'→ Need {12-1}x fewer inference steps with the full token')

---
## 16. Number of Possible Segmentations (Fibonacci)

For a string of length n, if all substrings up to length L are valid tokens:
$$|\mathcal{S}(n)| \leq F_{n+1} \quad \text{(when } L \geq 2\text{)}$$

Grows **exponentially** — this is why we need DP (Viterbi), not brute force.

In [ ]:
# --- Segmentation counting ---

def count_segmentations(n, max_token_len):
    """Count ways to segment a string of length n into tokens of length 1..L."""
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        for l in range(1, min(max_token_len, i) + 1):
            dp[i] += dp[i - l]
    return dp[n]

def fibonacci(n):
    """Return Fibonacci F(n)."""
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return a

print('Number of segmentations |S(n)| for string length n:\n')
print(f'{"n":>4} {"L=2":>12} {"F(n+1)":>12} {"L=3":>14} {"L=5":>14} {"L=10":>16} {"L=64":>18}')
print('-' * 90)

for n in [1, 2, 5, 10, 15, 20, 25, 30, 40, 50]:
    s2 = count_segmentations(n, 2)
    fn = fibonacci(n + 1)
    s3 = count_segmentations(n, 3)
    s5 = count_segmentations(n, 5)
    s10 = count_segmentations(n, 10)
    s64 = count_segmentations(n, min(64, n))
    print(f'{n:>4} {s2:>12,} {fn:>12,} {s3:>14,} {s5:>14,} {s10:>16,} {s64:>18,}')

print(f'\n→ L=2: |S(n)| = Fibonacci(n+1) — exponential growth (φ^n ≈ 1.618^n)')
print(f'→ L=64 (typical BPE max): ASTRONOMICAL for n > 20')
print(f'→ Viterbi DP: O(n × L) — linear in n, handles any L')
print(f'→ For n=50, L=64: {count_segmentations(50, 50):,} segmentations vs Viterbi\'s {50*64:,} operations')

---
## 17. Real Tokenizer Comparison (tiktoken)

Using OpenAI's `tiktoken` library to see real-world behavior.

In [ ]:
# --- Real tokenizer demo ---

try:
    import tiktoken
    HAS_TIKTOKEN = True
except ImportError:
    HAS_TIKTOKEN = False
    print('tiktoken not installed. Install with: pip install tiktoken')
    print('Showing pre-computed results instead.\n')

if HAS_TIKTOKEN:
    enc = tiktoken.get_encoding('cl100k_base')  # GPT-4 tokenizer
    
    test_texts = [
        "Hello, world!",
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is transforming artificial intelligence.",
        "E = mc²",
        "12345 + 67890 = 80235",
        "3.14159265358979",
        "こんにちは世界",          # Japanese
        "Привет мир",             # Russian
        "مرحبا بالعالم",            # Arabic
        "สวัสดีครับ",               # Thai
        "def fibonacci(n):\n    if n <= 1:\n        return n",  # code
    ]
    
    print(f'GPT-4 tokenizer (cl100k_base), V = {enc.n_vocab:,}\n')
    print(f'{"Text":<50} {"Tokens":>7} {"ρ":>6}')
    print('-' * 68)
    for text in test_texts:
        ids = enc.encode(text)
        rho = len(text) / len(ids)
        decoded = [enc.decode([i]) for i in ids]
        display = text[:47] + '...' if len(text) > 50 else text
        print(f'{display:<50} {len(ids):>7} {rho:>6.2f}')
        print(f'  → {decoded}')
    
    # Arithmetic tokenization deep-dive
    print(f'\n--- Arithmetic tokenization ---')
    for num in ['1', '12', '123', '1234', '12345', '123456', '1234567']:
        ids = enc.encode(num)
        decoded = [enc.decode([i]) for i in ids]
        print(f'  "{num}" → {decoded} ({len(ids)} tokens)')
    
    print(f'\n→ Numbers split INCONSISTENTLY: "123" may be 1 token but "1234" is 2')
    print(f'→ The model must learn that "4" after "123" means thousands place')
    print(f'→ This is a fundamental limitation of subword tokenization for arithmetic')

else:
    print('Pre-computed examples (GPT-4, cl100k_base, V=100,277):')
    examples = [
        ('"Hello, world!"',          4, '3.25', ["Hello", ",", " world", "!"]),
        ('"Machine learning..."',    6, '8.33', ["Machine", " learning", " is", " transform", "ing", "..."]),
        ('"12345 + 67890 = 80235"',  8, '2.63', ["123", "45", " +", " ", "678", "90", " =", " 80235"]),
        ('"こんにちは世界"',           5, '1.40', ["こん", "にち", "は", "世", "界"]),
    ]
    print(f'{"Text":<35} {"Tokens":>7} {"ρ":>6}  Segmentation')
    print('-' * 80)
    for text, n, rho, segs in examples:
        print(f'{text:<35} {n:>7} {rho:>6}  {segs}')

In [ ]:
# --- Cross-language fertility (tokenization tax) ---

if HAS_TIKTOKEN:
    # Same semantic content in different languages
    sentences = {
        'English':    'The cat sat on the mat.',
        'Spanish':    'El gato se sentó en la alfombra.',
        'French':     'Le chat s\'est assis sur le tapis.',
        'German':     'Die Katze saß auf der Matte.',
        'Chinese':    '猫坐在垫子上。',
        'Japanese':   '猫はマットの上に座った。',
        'Korean':     '고양이가 매트 위에 앉았다.',
        'Arabic':     'جلست القطة على السجادة.',
        'Hindi':      'बिल्ली चटाई पर बैठ गई।',
        'Thai':       'แมวนั่งบนเสื่อ',
    }
    
    en_tokens = len(enc.encode(sentences['English']))
    
    print(f'Same semantic content across languages (cl100k_base):\n')
    print(f'{"Language":<12} {"Chars":>6} {"Bytes":>7} {"Tokens":>7} {"ρ":>6} {"Tax":>6} {"Cost vs EN":>10}')
    print('-' * 60)
    
    for lang, text in sentences.items():
        ids = enc.encode(text)
        n_tokens = len(ids)
        n_bytes = len(text.encode('utf-8'))
        rho = len(text) / n_tokens
        tax = n_tokens / en_tokens
        bar = '█' * int(tax * 8)
        print(f'{lang:<12} {len(text):>6} {n_bytes:>7} {n_tokens:>7} {rho:>6.2f} {tax:>5.1f}x {bar}')
    
    print(f'\n→ Non-English languages pay a "tokenization tax"')
    print(f'→ CJK: 2-3x more tokens for same semantic content')
    print(f'→ Effects: higher API cost, shorter effective context, potentially lower quality')
    print(f'→ Root cause: English-dominated training data for BPE merge statistics')
    print(f'\n--- Dollar impact ---')
    price_per_1M = 2.50  # typical GPT-4 input pricing
    for lang in ['English', 'Japanese', 'Thai']:
        ids = enc.encode(sentences[lang])
        cost = len(ids) / 1_000_000 * price_per_1M
        print(f'  {lang}: {len(ids)} tokens → ${cost*1_000_000:.2f}/M tokens')
else:
    print('Install tiktoken for cross-language analysis: pip install tiktoken')

---
## Summary

| # | Concept | Key Formula | Why It Matters |
|---|---------|-------------|----------------|
| 1 | BPE merge | $(a^*,b^*) = \arg\max$ freq(a,b) | Greedy compression; GPT, LLaMA |
| 2 | Compression ratio | $\rho = \text{chars}/\text{tokens}$ | Context window efficiency |
| 3 | Unigram EM | $P(t) \leftarrow E[\text{count}(t)] / \sum E[\text{count}]$ | Probabilistic, globally better |
| 4 | Token pruning | $\Delta\mathcal{L}(t) = \mathcal{L}(V \setminus \{t\}) - \mathcal{L}(V)$ | Remove least-useful tokens |
| 5 | WordPiece | PMI = $\log \frac{P(ab)}{P(a)P(b)}$ | Merge surprising pairs |
| 6 | Context window | $\text{chars} = \text{window} \times \rho$ | CJK gets half the context |
| 7 | Embedding cost | $N \times d$ params | Vocab size → model cost |
| 8 | Fertility | $|T(w)|$ tokens/word | High fertility → expensive |
| 9 | Shannon entropy | $H = -\sum P \log P$ | Vocab utilization |
| 10 | Rényi entropy | $H_\alpha = \frac{1}{1-\alpha} \log \sum P^\alpha$ | Vocabulary balance |
| 11 | Shannon bound | $\text{bits/char} = H/\rho$ | Compression limit |
| 12 | Viterbi DP | $\text{dp}[j] = \min(\text{dp}[i] + w)$ | Optimal segmentation |
| 13 | Segmentation count | $|\mathcal{S}(n)| = F_{n+1}$ | Why DP is necessary |

**Next:** [Embedding Space Math](../02-Embedding-Space-Math/notes.md) — mapping token IDs to continuous vectors in $\mathbb{R}^d$.